# 03 — Data Cleaning & Loading

This notebook implements the **Data Preparation** phase of the CRISP-DM workflow for the
Divvy bike-share project. It picks up where `01_data_collection.ipynb` leaves off (a folder of
30+ raw monthly `*-divvy-tripdata.csv` files) and prepares them for the star-schema warehouse
created in `02_database_design.ipynb` (`divvy.dim_date`, `divvy.dim_station`, `divvy.dim_ride_type`,
`divvy.dim_member_type`, `divvy.fact_trip`).

For each raw file, the pipeline:

1. **Loads** the CSV with explicit dtypes so schema drift across 30+ months of exports doesn't
   silently break things.
2. **Cleans** it — trims whitespace, drops unusable rows (missing `ride_id`, unparseable
   timestamps, duplicate `ride_id`s, non-positive durations that would violate the database's
   `CHECK` constraints).
3. **Engineers features** required by `fact_trip` — `duration_minutes`, `start_hour`,
   `day_of_week`, `month_partition`, `start_date_key`/`end_date_key`, `is_round_trip`, and an
   `is_anomalous` quality flag (very short/long trips, out-of-bounds coordinates) — flagged
   rather than dropped, matching the schema's intent.
4. **Builds and upserts dimension rows** (dates, stations, ride types, member types) so natural
   keys are resolved to surrogate keys.
5. **Loads the fact rows** in bulk via `COPY` into a staging table, then
   `INSERT ... ON CONFLICT (ride_id) DO NOTHING` into `fact_trip` — safe to re-run on the same
   files without creating duplicates (idempotent).

The pipeline processes **one file at a time** (rather than concatenating 30+ files into a single
in-memory DataFrame) so memory usage stays flat regardless of how many months you throw at it,
and a bad file doesn't take down the whole batch — errors are caught, logged, and rolled back
per file.

### Import Libraries

In [ ]:
import glob
import logging
import os
from datetime import datetime
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
import psycopg2.extras as pg_extras
from tqdm import tqdm

import config  # same config.py used in 02_database_design.ipynb (database.ini reader)

### Configuration

In [ ]:
# Folder containing the 30+ raw monthly CSVs produced by 01_data_collection.ipynb
BASE_FOLDER = r"G:\\My Drive\\divvy_tripdata"
CSV_FOLDER = os.path.join(BASE_FOLDER, "csv_master")

# Glob pattern matching Divvy's monthly export naming convention, e.g. 202401-divvy-tripdata.csv
CSV_PATTERN = "*-divvy-tripdata.csv"

# Raw columns every Divvy monthly export is expected to contain
RAW_COLUMNS = [
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id", "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng", "member_casual",
]
STRING_COLS = [
    "ride_id", "rideable_type", "start_station_name", "start_station_id",
    "end_station_name", "end_station_id", "member_casual",
]

# Loose Chicagoland bounding box -- used only to FLAG geospatial outliers, never to drop rows
LAT_RANGE = (41.55, 42.15)
LNG_RANGE = (-88.10, -87.40)

FLAG_SHORT_SECONDS = 60   # trips under 60s are likely false starts / immediate redocks
FLAG_LONG_HOURS = 24      # trips over 24h are likely lost, stolen, or improperly docked bikes

# Where to write the run's data-quality report
REPORT_PATH = os.path.join(BASE_FOLDER, "etl_data_quality_report.csv")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(BASE_FOLDER, "etl_pipeline.log")),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("divvy_etl")

### Database Connection

Reuses the same `config.py` / `database.ini` pattern established in `02_database_design.ipynb`.
`autocommit` is left `False` here because each file is loaded inside its own transaction that we
either `COMMIT` (success) or `ROLLBACK` (any error), keeping the warehouse consistent even if the
batch is interrupted partway through.

In [ ]:
def connect_divvy():
    """Connect to the divvy_db database (mirrors 02_database_design.ipynb)."""
    conn = None
    try:
        params = config.config_divvy()
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        logger.error(f"Could not connect to divvy_db: {error}")
        return None

## Step 1 — Discover Source Files

In [ ]:
def discover_csv_files(csv_folder: str) -> list[Path]:
    """Find every raw monthly CSV, sorted chronologically by filename (YYYYMM prefix)."""
    paths = sorted(Path(csv_folder).glob(CSV_PATTERN))
    if not paths:
        raise FileNotFoundError(
            f"No files matching '{CSV_PATTERN}' found in {csv_folder}. "
            "Run 01_data_collection.ipynb first, or update CSV_FOLDER above."
        )
    logger.info(f"Discovered {len(paths)} source files in {csv_folder}")
    return paths

## Step 2 — Load & Clean

#### 2.1 Raw load

Explicit `dtype` on identifier/text columns prevents pandas from guessing a numeric dtype for a
column (e.g. `start_station_id`) in one month's file and a string dtype in another, which would
otherwise break the eventual merge/lookup logic silently.

In [ ]:
def load_raw_csv(path: Path) -> pd.DataFrame:
    """Read one Divvy monthly CSV with explicit dtypes."""
    dtype = {c: "string" for c in STRING_COLS}
    df = pd.read_csv(path, dtype=dtype)
    missing = set(RAW_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing expected columns: {missing}")
    df = df[RAW_COLUMNS].copy()
    df["__source_file"] = path.name
    return df

#### 2.2 Clean + feature-engineer

Rows are only **dropped** when they cannot possibly satisfy the database's constraints (missing
`ride_id`, unparseable timestamps, duplicate `ride_id` within the file, or `duration <= 0` which
would violate `chk_fact_trip_duration_positive` / `chk_fact_trip_start_before_end`). Everything
else that looks unusual (very short/long trips, coordinates outside the Chicago metro area) is
**flagged** via `is_anomalous` so it stays queryable rather than being silently discarded.

In [ ]:
def clean_trips(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Clean and feature-engineer a raw trips dataframe.

    Returns (clean_df, stats) where stats is a dict of QA counters used to build
    the end-of-run data-quality report.
    """
    stats = {"rows_in": len(df)}

    # Trim whitespace on string fields; normalize empty strings to NA
    for col in STRING_COLS:
        df[col] = df[col].str.strip()
        df.loc[df[col] == "", col] = pd.NA

    # Drop rows with no ride_id (can't dedupe / violates the fact table's PK)
    before = len(df)
    df = df[df["ride_id"].notna()]
    stats["dropped_missing_ride_id"] = before - len(df)

    # Drop exact duplicate ride_ids within this file (keep first occurrence)
    before = len(df)
    df = df.drop_duplicates(subset="ride_id", keep="first")
    stats["dropped_duplicate_ride_id"] = before - len(df)

    # Parse timestamps -- handles both "YYYY-MM-DD HH:MM:SS" and
    # "YYYY-MM-DD HH:MM:SS.ffffff" formats seen across different export months
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")
    before = len(df)
    df = df[df["started_at"].notna() & df["ended_at"].notna()]
    stats["dropped_unparseable_timestamps"] = before - len(df)

    # Duration must be strictly positive to satisfy the DB CHECK constraints. Round
    # to 2dp FIRST (matching the fact_trip numeric(10,2) column) since sub-hundredth-
    # of-a-minute trips (e.g. 0.285 seconds) are non-positive *after* rounding even
    # though the raw duration is technically > 0 -- rounding after filtering would
    # let a few 0.00-minute rows slip through and violate the DB CHECK constraint.
    duration_minutes = ((df["ended_at"] - df["started_at"]).dt.total_seconds() / 60.0).round(2)
    before = len(df)
    keep = duration_minutes > 0
    stats["dropped_non_positive_duration"] = int((~keep).sum())
    df = df[keep].copy()
    df["duration_minutes"] = duration_minutes[keep]

    # Defensive fill so NOT NULL dimension foreign keys never break on ingest
    stats["filled_missing_rideable_type"] = int(df["rideable_type"].isna().sum())
    stats["filled_missing_member_casual"] = int(df["member_casual"].isna().sum())
    df["rideable_type"] = df["rideable_type"].fillna("unknown")
    df["member_casual"] = df["member_casual"].fillna("unknown")

    # Coerce lat/lng to numeric (guards against stray text values)
    for col in ["start_lat", "start_lng", "end_lat", "end_lng"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Feature engineering required by divvy.fact_trip ---
    df["start_hour"] = df["started_at"].dt.hour.astype("int64")
    df["day_of_week"] = df["started_at"].dt.dayofweek + 1  # 1=Monday .. 7=Sunday
    df["month_partition"] = (df["started_at"].dt.year * 100 + df["started_at"].dt.month).astype("int64")
    df["start_date_key"] = df["started_at"].dt.strftime("%Y%m%d").astype("int64")
    df["end_date_key"] = df["ended_at"].dt.strftime("%Y%m%d").astype("int64")

    has_both_stations = df["start_station_id"].notna() & df["end_station_id"].notna()
    df["is_round_trip"] = pd.NA
    df.loc[has_both_stations, "is_round_trip"] = (
        df.loc[has_both_stations, "start_station_id"] == df.loc[has_both_stations, "end_station_id"]
    )

    out_of_bounds = (
        ~df["start_lat"].between(*LAT_RANGE) | ~df["start_lng"].between(*LNG_RANGE)
        | (df["end_lat"].notna() & ~df["end_lat"].between(*LAT_RANGE))
        | (df["end_lng"].notna() & ~df["end_lng"].between(*LNG_RANGE))
    )
    too_short = df["duration_minutes"] < (FLAG_SHORT_SECONDS / 60.0)
    too_long = df["duration_minutes"] > (FLAG_LONG_HOURS * 60)
    df["is_anomalous"] = (out_of_bounds | too_short | too_long).fillna(False)

    stats["flagged_anomalous"] = int(df["is_anomalous"].sum())
    stats["flagged_short_trips"] = int(too_short.sum())
    stats["flagged_long_trips"] = int(too_long.sum())
    stats["flagged_out_of_bounds_coords"] = int(out_of_bounds.fillna(False).sum())
    stats["rows_out"] = len(df)

    return df, stats

## Step 3 — Build Dimension Rows

Station names occasionally change over time for the same `station_id` (renames, rebranding).
The **most frequently occurring name** for each `station_id` is used as the canonical
`station_name`, and the **median** lat/lng is used as its canonical location -- both computed
per-file and then reconciled across files by the `ON CONFLICT` upsert in Step 4.

In [ ]:
def build_dim_date(clean_df: pd.DataFrame) -> pd.DataFrame:
    """Generate a full calendar covering every start/end date seen in this batch."""
    min_d = min(clean_df["started_at"].min(), clean_df["ended_at"].min()).normalize()
    max_d = max(clean_df["started_at"].max(), clean_df["ended_at"].max()).normalize()
    dates = pd.date_range(min_d, max_d, freq="D")
    dim_date = pd.DataFrame({"full_date": dates})
    dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype("int64")
    dim_date["year"] = dim_date["full_date"].dt.year
    dim_date["quarter"] = dim_date["full_date"].dt.quarter
    dim_date["month"] = dim_date["full_date"].dt.month
    dim_date["day"] = dim_date["full_date"].dt.day
    dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1  # 1=Monday .. 7=Sunday
    dim_date["day_name"] = dim_date["full_date"].dt.day_name()
    dim_date["month_name"] = dim_date["full_date"].dt.month_name()
    dim_date["is_weekend"] = dim_date["day_of_week"].isin([6, 7])
    cols = ["date_key", "full_date", "year", "quarter", "month", "day",
            "day_of_week", "day_name", "month_name", "is_weekend"]
    return dim_date[cols]


def build_dim_station(clean_df: pd.DataFrame) -> pd.DataFrame:
    """Resolve one canonical (name, lat, lng) row per station_id from both trip endpoints."""
    starts = clean_df[["start_station_id", "start_station_name", "start_lat", "start_lng"]].rename(
        columns={"start_station_id": "station_id", "start_station_name": "station_name",
                 "start_lat": "lat", "start_lng": "lng"})
    ends = clean_df[["end_station_id", "end_station_name", "end_lat", "end_lng"]].rename(
        columns={"end_station_id": "station_id", "end_station_name": "station_name",
                 "end_lat": "lat", "end_lng": "lng"})
    all_st = pd.concat([starts, ends], ignore_index=True)
    all_st = all_st[all_st["station_id"].notna()]

    def _mode_or_na(s):
        m = s.dropna().mode()
        return m.iat[0] if not m.empty else pd.NA

    name_mode = all_st.groupby("station_id")["station_name"].agg(_mode_or_na)
    coords = all_st.groupby("station_id")[["lat", "lng"]].median()
    dim_station = name_mode.to_frame().join(coords).reset_index()
    dim_station.columns = ["station_id", "station_name", "latitude", "longitude"]
    return dim_station

## Step 4 — Load to PostgreSQL

Dimension tables are tiny (dozens to a few thousand rows even after 30+ months), so a simple
`execute_values(... ON CONFLICT ...)` upsert is fast enough. The fact table can reach tens of
millions of rows across a full multi-year history, so it uses the much faster
`COPY` **-into-staging-table -> `INSERT ... ON CONFLICT DO NOTHING`** pattern instead of row-by-row
inserts.

In [ ]:
def upsert_dim_date(conn, dim_date: pd.DataFrame) -> None:
    sql = """
        INSERT INTO divvy.dim_date
            (date_key, full_date, year, quarter, month, day, day_of_week, day_name, month_name, is_weekend)
        VALUES %s
        ON CONFLICT (date_key) DO NOTHING;
    """
    rows = list(dim_date.itertuples(index=False, name=None))
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, rows)


def upsert_dim_station(conn, dim_station: pd.DataFrame) -> None:
    sql = """
        INSERT INTO divvy.dim_station (station_id, station_name, latitude, longitude)
        VALUES %s
        ON CONFLICT (station_id) DO UPDATE
            SET station_name = COALESCE(EXCLUDED.station_name, divvy.dim_station.station_name),
                latitude     = COALESCE(EXCLUDED.latitude, divvy.dim_station.latitude),
                longitude    = COALESCE(EXCLUDED.longitude, divvy.dim_station.longitude);
    """
    rows = [tuple(None if pd.isna(v) else v for v in r)
            for r in dim_station.itertuples(index=False, name=None)]
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, rows)


def upsert_dim_ride_type(conn, values: list[str]) -> None:
    sql = "INSERT INTO divvy.dim_ride_type (rideable_type) VALUES %s ON CONFLICT (rideable_type) DO NOTHING;"
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, [(v,) for v in values])


def upsert_dim_member_type(conn, values: list[str]) -> None:
    sql = "INSERT INTO divvy.dim_member_type (member_casual) VALUES %s ON CONFLICT (member_casual) DO NOTHING;"
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, [(v,) for v in values])


def fetch_key_maps(conn) -> dict:
    """Pull the current natural-key -> surrogate-key mappings back from the warehouse."""
    with conn.cursor() as cur:
        cur.execute("SELECT station_id, station_key FROM divvy.dim_station")
        station = pd.DataFrame(cur.fetchall(), columns=["station_id", "station_key"])
        cur.execute("SELECT rideable_type, ride_type_key FROM divvy.dim_ride_type")
        ride_type = pd.DataFrame(cur.fetchall(), columns=["rideable_type", "ride_type_key"])
        cur.execute("SELECT member_casual, member_type_key FROM divvy.dim_member_type")
        member_type = pd.DataFrame(cur.fetchall(), columns=["member_casual", "member_type_key"])
    return {"station": station, "ride_type": ride_type, "member_type": member_type}

In [ ]:
def build_fact_dataframe(clean_df: pd.DataFrame, key_maps: dict) -> pd.DataFrame:
    """Resolve natural keys to surrogate keys and assemble the fact_trip-shaped dataframe."""
    fact = clean_df.merge(
        key_maps["station"], left_on="start_station_id", right_on="station_id", how="left"
    ).rename(columns={"station_key": "start_station_key"}).drop(columns=["station_id"])

    fact = fact.merge(
        key_maps["station"], left_on="end_station_id", right_on="station_id", how="left"
    ).rename(columns={"station_key": "end_station_key"}).drop(columns=["station_id"])

    fact = fact.merge(key_maps["ride_type"], on="rideable_type", how="left")
    fact = fact.merge(key_maps["member_type"], on="member_casual", how="left")

    # ride_type_key / member_type_key are NOT NULL on fact_trip -- if a lookup fails
    # (e.g. dimension upsert happened out of order), fail loudly rather than silently
    # loading bad rows.
    n_missing = fact["ride_type_key"].isna().sum() + fact["member_type_key"].isna().sum()
    if n_missing:
        raise ValueError(f"{n_missing} rows failed to resolve ride_type_key/member_type_key")

    fact_cols = [
        "ride_id", "start_date_key", "end_date_key", "start_station_key", "end_station_key",
        "ride_type_key", "member_type_key", "started_at", "ended_at", "duration_minutes",
        "start_hour", "day_of_week", "month_partition",
        "start_lat", "start_lng", "end_lat", "end_lng",
        "is_round_trip", "is_anomalous",
    ]
    fact_final = fact[fact_cols].copy()
    fact_final["is_round_trip"] = fact_final["is_round_trip"].astype("object").where(
        fact_final["is_round_trip"].notna(), None
    )

    # Merges introduce float NaN for unmatched keys, which silently upcasts integer
    # columns to float64 (e.g. 820 -> "820.0"). Postgres' integer COPY rejects "820.0",
    # so cast every integer-typed fact column to pandas' nullable Int64 to preserve
    # NULLs without adding a decimal point.
    int_cols = [
        "start_date_key", "end_date_key", "start_station_key", "end_station_key",
        "ride_type_key", "member_type_key", "start_hour", "day_of_week", "month_partition",
    ]
    for col in int_cols:
        fact_final[col] = fact_final[col].astype("Int64")

    return fact_final


def load_fact_trip(conn, fact_df: pd.DataFrame) -> int:
    """Bulk-load fact rows via COPY into a temp staging table, then upsert into fact_trip.

    Returns the number of NEW rows actually inserted (existing ride_ids are skipped).
    """
    fact_cols = list(fact_df.columns)
    with conn.cursor() as cur:
        cur.execute(f"""
            CREATE TEMP TABLE staging_fact_trip (LIKE divvy.fact_trip INCLUDING DEFAULTS)
            ON COMMIT DROP;
            ALTER TABLE staging_fact_trip DROP COLUMN trip_key;
        """)

        buf = StringIO()
        fact_df.to_csv(buf, index=False, header=False, na_rep="\\N")
        buf.seek(0)
        cur.copy_expert(
            f"COPY staging_fact_trip ({', '.join(fact_cols)}) FROM STDIN WITH (FORMAT csv, NULL '\\N')",
            buf,
        )

        cur.execute(f"""
            INSERT INTO divvy.fact_trip ({', '.join(fact_cols)})
            SELECT {', '.join(fact_cols)} FROM staging_fact_trip
            ON CONFLICT (ride_id) DO NOTHING;
        """)
        return cur.rowcount

## Step 5 — Orchestration

Each file is processed inside its own transaction: dimension upserts and the fact load either
all succeed and `COMMIT`, or any failure triggers a `ROLLBACK` and the file is logged as failed --
subsequent files are still processed. This makes the pipeline **safe to re-run** on the same
folder (e.g. after fixing one bad file, or after downloading new months).

In [ ]:
def process_file(path: Path, conn) -> dict:
    """Run the full clean -> dimension-upsert -> fact-load flow for a single CSV file."""
    result = {"file": path.name, "status": "success", "error": None}
    try:
        raw = load_raw_csv(path)
        clean, clean_stats = clean_trips(raw)
        result.update(clean_stats)

        if clean.empty:
            result["status"] = "skipped_empty"
            result["fact_rows_inserted"] = 0
            return result

        dim_date = build_dim_date(clean)
        dim_station = build_dim_station(clean)
        ride_types = clean["rideable_type"].dropna().unique().tolist()
        member_types = clean["member_casual"].dropna().unique().tolist()

        upsert_dim_date(conn, dim_date)
        upsert_dim_station(conn, dim_station)
        upsert_dim_ride_type(conn, ride_types)
        upsert_dim_member_type(conn, member_types)

        key_maps = fetch_key_maps(conn)
        fact_df = build_fact_dataframe(clean, key_maps)
        inserted = load_fact_trip(conn, fact_df)

        conn.commit()
        result["fact_rows_inserted"] = inserted
        logger.info(f"{path.name}: cleaned {clean_stats['rows_out']:,} rows, inserted {inserted:,} new fact rows")

    except Exception as exc:
        conn.rollback()
        result["status"] = "failed"
        result["error"] = str(exc)
        logger.error(f"{path.name}: FAILED -- {exc}")

    return result

In [ ]:
def run_pipeline(csv_folder: str = CSV_FOLDER) -> pd.DataFrame:
    """Process every discovered CSV file and return a per-file summary report."""
    files = discover_csv_files(csv_folder)
    conn = connect_divvy()
    if conn is None:
        raise ConnectionError("Could not connect to divvy_db.")

    summary_rows = []
    try:
        for path in tqdm(files, desc="Cleaning & loading"):
            summary_rows.append(process_file(path, conn))
    finally:
        conn.close()

    report = pd.DataFrame(summary_rows)
    report.to_csv(REPORT_PATH, index=False)
    logger.info(f"Data-quality report written to {REPORT_PATH}")
    return report

## Step 6 — Run the Pipeline

In [ ]:
report = run_pipeline()

n_failed = (report["status"] == "failed").sum()
print(f"Files processed : {len(report)}")
print(f"Succeeded        : {(report['status'] == 'success').sum()}")
print(f"Failed           : {n_failed}")
print(f"Total rows in    : {report.get('rows_in', pd.Series(dtype=int)).sum():,}")
print(f"Total rows out   : {report.get('rows_out', pd.Series(dtype=int)).sum():,}")
print(f"Total fact rows inserted : {report.get('fact_rows_inserted', pd.Series(dtype=int)).sum():,}")

if n_failed:
    print("\nFailed files:")
    print(report.loc[report['status'] == 'failed', ['file', 'error']])

report

## Step 7 — Post-Load Validation

Quick sanity checks: row counts per table, referential integrity, and idempotency (re-running the pipeline on the same files should insert 0 new rows the second time).

In [ ]:
def validate_load():
    conn = connect_divvy()
    checks = {}
    try:
        with conn.cursor() as cur:
            for table in ["dim_date", "dim_station", "dim_ride_type", "dim_member_type", "fact_trip"]:
                cur.execute(f"SELECT COUNT(*) FROM divvy.{table};")
                checks[f"{table}_row_count"] = cur.fetchone()[0]

            cur.execute("""
                SELECT COUNT(*) FROM divvy.fact_trip f
                LEFT JOIN divvy.dim_station s ON f.start_station_key = s.station_key
                WHERE f.start_station_key IS NOT NULL AND s.station_key IS NULL;
            """)
            checks["orphan_start_station_refs"] = cur.fetchone()[0]

            cur.execute("""
                SELECT COUNT(*) FROM divvy.fact_trip f
                LEFT JOIN divvy.dim_date d ON f.start_date_key = d.date_key
                WHERE d.date_key IS NULL;
            """)
            checks["orphan_start_date_refs"] = cur.fetchone()[0]

            cur.execute("SELECT MIN(started_at), MAX(started_at) FROM divvy.fact_trip;")
            checks["date_range_loaded"] = cur.fetchone()
    finally:
        conn.close()
    return checks

validate_load()